# Fuel Cost-Shock Forecaster — Backtest & Eval

Companion to [`01_fuel_shock_multi_agent.ipynb`](01_fuel_shock_multi_agent.ipynb), which defines
and demos the multi-agent pipeline at a single live origin. **This notebook is where the
pipeline actually gets backtested** — `01_...` builds the `data_service` / agent configs / task
specs but never calls the evaluation harness against historical origins; this notebook does.

**Why a separate notebook, not new cells in `01_...`.** `01_...` is kept exactly as-is (the
demo/reference notebook); all backtest-running code lives here instead, re-declaring the setup
it needs from `01_...` inline (per this project's notebook-only convention — no shared `.py`
module between the two notebooks).

**The leakage question this notebook exists to answer.** Both configured models
(`gemini-3.1-flash-lite-preview`, `gemini-3.5-flash`) have a **~January 2025** knowledge cutoff.
Backtesting before that date would let the Forecaster Agent's binary shock probability simply
*recall* what happened rather than *predict* it — a false-positive on accuracy. Every origin
evaluated below is in **`specs/fuel_shock_backtest.yaml`'s June 2025 – August 2026 window**,
comfortably after the cutoff. On top of the window choice, the News Agent's `search_web` tool
has its own code-level temporal fence (not just a prompt instruction) that stops it from
grounding on post-origin news during a backtest — see §8 below for how to see it in action.

**Sections:**
1. Setup
2. Commodity-Data Agent (re-declared from `01_...`)
3. Geopolitical/News Agent (re-declared from `01_...`)
4. Forecaster Agent — with a backtest-safe prompt builder
5. Baselines: historical frequency, GARCH(1,1), logistic regression
6. Smoke test (3 origins)
7. Full post-cutoff backtest (~14 origins)
8. Leaderboard & calibration
9. Leakage-fence spot check
10. Caveats and next steps


In [ ]:
import json
import logging
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yaml
from IPython.display import Markdown, display  # noqa: A004

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters.base import BaseAdapter
from aieng.forecasting.data.adapters.fred import FREDAdapter
from aieng.forecasting.data.adapters.yfinance import YFinanceDailyAdapter
from aieng.forecasting.data.context import ForecastContext
from aieng.forecasting.evaluation.backtest import BacktestSpec
from aieng.forecasting.evaluation.artifacts import cached_backtest
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.methods import GARCHPredictor, HistoricalFrequencyPredictor, LogisticRegressionBaseline
from aieng.forecasting.methods.agentic import AgentPredictor, DiscreteAgentForecastOutput
from aieng.forecasting.methods.agentic.agent_factory import AgentConfig, ContextRetrievalConfig
from aieng.forecasting.models import ADVANCED_MODEL, LITE_MODEL
from dotenv import load_dotenv
from pydantic import BaseModel


warnings.filterwarnings("ignore")
load_dotenv()

# -- Model selection (same as 01_fuel_shock_multi_agent.ipynb) --------------------------------
AGENT_MODEL = LITE_MODEL
SEARCH_MODEL = LITE_MODEL
VERIFIER_MODEL = ADVANCED_MODEL

# -- Cost-shock target definition (same as 01_fuel_shock_multi_agent.ipynb) -------------------
SHOCK_THRESHOLD_PCT = 0.10
SHOCK_HORIZON_DAYS = 21  # ~1 trading month


def naive_utc_now() -> datetime:
    """Timezone-naive current UTC time (DataService / CutoffEnforcer require this)."""
    return datetime.now(tz=timezone.utc).replace(tzinfo=None)


def repo_root() -> Path:
    """Walk up from CWD to find the repo root (the directory containing pyproject.toml)."""
    cwd = Path.cwd().resolve()
    root = cwd
    while not (root / "pyproject.toml").exists():
        if root.parent == root:
            return cwd
        root = root.parent
    return root


ROOT = repo_root()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
SPECS_DIR = ROOT / "implementations" / "fuel_cost_shock_forecaster" / "specs"
PREDICTIONS_DIR = DATA_DIR / "predictions"
print(f"Repo root: {ROOT}")
print(f"Specs dir: {SPECS_DIR}")


---
## 1. Commodity-Data Agent (re-declared from `01_...`)

Verbatim from `01_fuel_shock_multi_agent.ipynb` — the `DataService` registration layer covering
yfinance (jet-fuel proxy, WTI, USD index), FRED (CAD/USD FX), EIA (crude/jet-fuel spot), and the
derived cost-shock event series. See `01_...` §1 for the full rationale; this notebook only needs
it re-declared so it can build historical `ForecastContext`s for the backtest harness.


In [ ]:
# -- EIA Open Data adapter (API v2) -- verbatim from 01_fuel_shock_multi_agent.ipynb --------
EIA_BASE_URL = "https://api.eia.gov/v2"
EIA_PAGE_LENGTH = 5000  # API max rows per request


class EIAAdapter(BaseAdapter):
    """Adapter for a single EIA Open Data API v2 petroleum series, with disk cache."""

    def __init__(
        self,
        route1: str,
        route2: str,
        series_id: str,
        *,
        frequency: str = "daily",
        api_key: str | None = None,
        cache_dir: Path | None = None,
        refresh: bool = False,
    ) -> None:
        self._route1 = route1
        self._route2 = route2
        self._series_id = series_id
        self._frequency = frequency
        self._api_key = api_key or os.environ.get("EIA_API_KEY")
        self._cache_dir = cache_dir if cache_dir is not None else DATA_DIR / "eia"
        self._refresh = refresh

    @property
    def cache_path(self) -> Path:
        return self._cache_dir / f"{self._series_id}.parquet"

    def fetch(self) -> pd.DataFrame:
        if self.cache_path.exists() and not self._refresh:
            df = pd.read_parquet(self.cache_path)
            df["timestamp"] = pd.to_datetime(df["timestamp"])
            df["released_at"] = pd.to_datetime(df["released_at"])
            return df

        df = self._fetch_from_api()
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(self.cache_path, index=False)
        return df

    def _fetch_from_api(self) -> pd.DataFrame:
        if not self._api_key:
            raise ValueError(
                "EIA API key not provided. Set the EIA_API_KEY environment variable "
                "or pass api_key= to EIAAdapter."
            )

        url = f"{EIA_BASE_URL}/petroleum/{self._route1}/{self._route2}/data/"
        rows: list[dict] = []
        offset = 0
        while True:
            params = {
                "api_key": self._api_key,
                "frequency": self._frequency,
                "data[0]": "value",
                "facets[series][]": self._series_id,
                "sort[0][column]": "period",
                "sort[0][direction]": "asc",
                "offset": offset,
                "length": EIA_PAGE_LENGTH,
            }
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()["response"]
            page = payload["data"]
            rows.extend(page)
            if len(page) < EIA_PAGE_LENGTH:
                break
            offset += EIA_PAGE_LENGTH

        if not rows:
            raise RuntimeError(f"EIA series '{self._series_id}' returned no data.")

        df = pd.DataFrame(rows)
        df["timestamp"] = pd.to_datetime(df["period"])
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df = df.dropna(subset=["value"]).sort_values("timestamp").reset_index(drop=True)
        df["released_at"] = df["timestamp"]  # EIA doesn't expose vintage dates via this route
        return df[["timestamp", "value", "released_at"]]

    def __repr__(self) -> str:
        return f"EIAAdapter(route={self._route1}/{self._route2}, series_id={self._series_id!r})"


print("EIAAdapter defined.")


In [ ]:
# -- Canonical series IDs -- verbatim from 01_fuel_shock_multi_agent.ipynb -------------------
FUEL_SERIES_ID = "jet_fuel_proxy_price"  # yfinance HO=F -- primary target
WTI_SERIES_ID = "wti_crude_oil_price"  # yfinance CL=F -- covariate
USD_INDEX_SERIES_ID = "usd_index_price"  # yfinance DX-Y.NYB -- covariate
CAD_USD_FX_SERIES_ID = "cad_usd_fx_rate"  # FRED DEXCAUS -- covariate
EIA_WTI_SPOT_SERIES_ID = "eia_wti_spot_price"  # EIA RWTC -- covariate
EIA_JET_FUEL_SPOT_SERIES_ID = "eia_jet_fuel_spot_price"  # EIA EER_EPJK_PF4_RGC_DPG -- covariate
SHOCK_EVENT_SERIES_ID = "fuel_cost_shock_event_21d"  # derived binary series -- primary task target

_FUEL_TICKER = "HO=F"
_WTI_TICKER = "CL=F"
_USD_INDEX_TICKER = "DX-Y.NYB"
_HISTORY_START = "2004-01-01"


def derive_shock_event_series(
    price_df: pd.DataFrame,
    *,
    horizon_days: int = SHOCK_HORIZON_DAYS,
    threshold_pct: float = SHOCK_THRESHOLD_PCT,
) -> pd.DataFrame:
    """Derive the rolling binary cost-shock event series from a daily price series."""
    df = price_df.sort_values("timestamp").reset_index(drop=True)
    timestamps = pd.to_datetime(df["timestamp"])
    values = df["value"].astype(float)

    n = len(df)
    if n <= horizon_days:
        return pd.DataFrame(columns=["timestamp", "value", "released_at"])

    origin = values.iloc[: n - horizon_days].reset_index(drop=True)
    resolved = values.iloc[horizon_days:].reset_index(drop=True)
    pct_change = resolved / origin - 1.0

    return pd.DataFrame(
        {
            "timestamp": timestamps.iloc[: n - horizon_days].reset_index(drop=True),
            "value": (pct_change > threshold_pct).astype(float),
            "released_at": timestamps.iloc[horizon_days:].reset_index(drop=True),
        }
    )


class FuelShockEventAdapter(BaseAdapter):
    """Adapter producing the derived rolling cost-shock event series."""

    def __init__(
        self, price_adapter: BaseAdapter, *, horizon_days: int = SHOCK_HORIZON_DAYS, threshold_pct: float = SHOCK_THRESHOLD_PCT
    ) -> None:
        self._price_adapter = price_adapter
        self._horizon_days = horizon_days
        self._threshold_pct = threshold_pct

    def fetch(self) -> pd.DataFrame:
        price_df = self._price_adapter.fetch()
        return derive_shock_event_series(price_df, horizon_days=self._horizon_days, threshold_pct=self._threshold_pct)


def build_fuel_service(cache_dir: Path | None = None) -> DataService:
    """Return a :class:`DataService` with the fuel cost-shock series registered."""
    resolved_cache_dir = cache_dir if cache_dir is not None else DATA_DIR / "yfinance"
    svc = DataService()

    fuel_adapter = YFinanceDailyAdapter(ticker=_FUEL_TICKER, start=_HISTORY_START, cache_dir=resolved_cache_dir)
    svc.register(
        FUEL_SERIES_ID,
        fuel_adapter,
        SeriesMetadata(
            series_id=FUEL_SERIES_ID,
            description="NY Harbor ULSD / Heating Oil front-month futures adjusted close (Yahoo Finance HO=F) -- jet-fuel proxy.",
            source="yfinance",
            units="USD/gal",
            frequency="B",
        ),
    )

    svc.register(
        SHOCK_EVENT_SERIES_ID,
        FuelShockEventAdapter(fuel_adapter),
        SeriesMetadata(
            series_id=SHOCK_EVENT_SERIES_ID,
            description=(
                f"Cost-shock indicator: 1.0 if {FUEL_SERIES_ID} rises more than "
                f"{SHOCK_THRESHOLD_PCT:.0%} over the following {SHOCK_HORIZON_DAYS} business days, else 0.0."
            ),
            source=f"Derived ({FUEL_SERIES_ID})",
            units="0/1 event indicator",
            frequency="B",
        ),
    )

    svc.register(
        WTI_SERIES_ID,
        YFinanceDailyAdapter(ticker=_WTI_TICKER, start=_HISTORY_START, cache_dir=resolved_cache_dir),
        SeriesMetadata(
            series_id=WTI_SERIES_ID,
            description="WTI Crude Oil front-month futures adjusted close (Yahoo Finance CL=F)",
            source="yfinance",
            units="USD/bbl",
            frequency="B",
        ),
    )

    svc.register(
        USD_INDEX_SERIES_ID,
        YFinanceDailyAdapter(ticker=_USD_INDEX_TICKER, field="Adj Close", start=_HISTORY_START, cache_dir=resolved_cache_dir),
        SeriesMetadata(
            series_id=USD_INDEX_SERIES_ID,
            description="US Dollar Index close level (Yahoo Finance DX-Y.NYB)",
            source="yfinance",
            units="index-level",
            frequency="B",
        ),
    )

    try:
        svc.register(
            CAD_USD_FX_SERIES_ID,
            FREDAdapter("DEXCAUS", cache_dir=DATA_DIR / "fred"),
            SeriesMetadata(
                series_id=CAD_USD_FX_SERIES_ID,
                description="Canadian Dollars to U.S. Dollar spot exchange rate (FRED DEXCAUS)",
                source="FRED (DEXCAUS)",
                units="CAD per USD",
                frequency="B",
            ),
        )
    except (RuntimeError, ValueError) as exc:
        warnings.warn(f"Skipping CAD/USD FX (FRED): {exc}", stacklevel=2)

    try:
        svc.register(
            EIA_WTI_SPOT_SERIES_ID,
            EIAAdapter("pri", "spt", "RWTC"),
            SeriesMetadata(
                series_id=EIA_WTI_SPOT_SERIES_ID,
                description="Cushing, OK WTI spot price FOB (EIA RWTC)",
                source="EIA Open Data (petroleum/pri/spt, RWTC)",
                units="USD/bbl",
                frequency="B",
            ),
        )
        svc.register(
            EIA_JET_FUEL_SPOT_SERIES_ID,
            EIAAdapter("pri", "spt", "EER_EPJK_PF4_RGC_DPG"),
            SeriesMetadata(
                series_id=EIA_JET_FUEL_SPOT_SERIES_ID,
                description="US Gulf Coast kerosene-type jet-fuel spot price FOB (EIA EER_EPJK_PF4_RGC_DPG)",
                source="EIA Open Data (petroleum/pri/spt, EER_EPJK_PF4_RGC_DPG)",
                units="USD/gal",
                frequency="B",
            ),
        )
    except (RuntimeError, ValueError) as exc:
        warnings.warn(f"Skipping EIA spot-price covariates: {exc}", stacklevel=2)

    return svc


data_service = build_fuel_service()
ctx0 = data_service.context(as_of=naive_utc_now())
fuel_df = ctx0.get_series(FUEL_SERIES_ID)
event_df = ctx0.get_series(SHOCK_EVENT_SERIES_ID)
print(f"Jet-fuel proxy (HO=F) history through {pd.Timestamp(fuel_df['timestamp'].max()).date()}, {len(fuel_df)} rows")
print(f"Resolved shock-event history: {len(event_df)} rows, base rate = {event_df['value'].mean():.1%}")


---
## 2. Geopolitical/News Agent (re-declared from `01_...`)

Verbatim from `01_fuel_shock_multi_agent.ipynb` §2. The temporal-leakage fence this section wires
in (`search_web`'s harness-controlled `as_of` + independent verifier model) is exactly the
mechanism that makes backtesting *before* today's date safe from news leakage — see §8 for how to
watch it fire during the backtest below.


In [ ]:
FUEL_NEWS_INSTRUCTION = """
You are a jet-fuel and crude-oil market intelligence specialist with access to web search.

Search for information relevant to the query and return a concise structured markdown summary (3-5 paragraphs) covering relevant aspects of:
- Crude oil (WTI/Brent) and refined-products (jet fuel / heating oil / diesel) price level and recent trend
- OPEC+ production decisions and supply outlook
- Geopolitical risks in the Persian Gulf, Middle East, Strait of Hormuz, and other key shipping lanes affecting crude and product tanker flows
- US EIA policy releases, Strategic Petroleum Reserve actions, and inventory data surprises
- Refinery outages or disruptions affecting jet fuel / distillate supply
- Notable analyst forecasts or unusual price-target revisions for crude or refined products

Ground your summary in the search results you actually retrieve. When a cutoff date is specified, do not report or speculate about events that occurred after that date.

Before finalizing your summary, reason step by step: (1) for each candidate fact, judge its actual recency from the substance of the result itself, never from a source's claimed publish date or byline timestamp -- those are frequently stale or updated after original publication; (2) discard anything you cannot confidently place before the cutoff date; (3) only then write your summary. Do not supplement the search results with your own background/training knowledge -- if the results are insufficient, say so explicitly rather than filling gaps from memory.
""".strip()

FUEL_CONTEXT_RETRIEVAL_SUPPLEMENT = """

## Context retrieval

Call ``search_web`` to gather market intelligence BEFORE producing forecasts.

Call ``search_web`` with ``query`` and ``cutoff_date`` (set to the ``as_of`` date from the payload). The ``cutoff_date`` MUST always equal ``as_of`` -- this is the temporal fence that prevents post-origin information from contaminating historical backtests.

If ``search_web`` returns a result beginning with ``[SEARCH_VERIFICATION_FAILED]``, treat it as no verified news context for that query. Do not use your own background knowledge to fill the gap or speculate about what the news might have said -- proceed with price-history and other available signals only, and note the gap in your rationale.

Recommended queries (call ``search_web`` once per topic):
- ``search_web(query="crude oil and jet fuel price trend and OPEC+ supply decisions", cutoff_date=<as_of>)``
- ``search_web(query="Persian Gulf geopolitical risk shipping lane disruptions", cutoff_date=<as_of>)``
- ``search_web(query="US EIA policy releases and Strategic Petroleum Reserve actions", cutoff_date=<as_of>)``
- ``search_web(query="refinery outages affecting jet fuel and distillate supply", cutoff_date=<as_of>)``
"""


def build_fuel_context_retrieval_config(
    search_model: str = SEARCH_MODEL,
    verifier_model: str = VERIFIER_MODEL,
    verifier_max_attempts: int = 3,
    verifier_confidence_threshold: int = 8,
) -> ContextRetrievalConfig:
    """Build the Geopolitical/News Agent's :class:`ContextRetrievalConfig`."""
    return ContextRetrievalConfig(
        enabled=True,
        instruction=FUEL_NEWS_INSTRUCTION,
        search_model=search_model,
        verifier_model=verifier_model,
        verifier_max_attempts=verifier_max_attempts,
        verifier_confidence_threshold=verifier_confidence_threshold,
    )


print("News Agent instruction + factory defined.")


---
## 3. Forecaster Agent — with a backtest-safe prompt builder

Mostly verbatim from `01_fuel_shock_multi_agent.ipynb` §3-4, with **one deliberate fix**:
`01_...`'s `FuelMultitaskPromptBuilder.__call__` reads price history via
`context.get_series(task.target_series_id)`. That works for `01_...`'s own live demo, where the
demo tasks are hand-built with `target_series_id=FUEL_SERIES_ID` (the price series). It does
**not** work against `specs/fuel_shock_backtest.yaml`, whose `task.target_series_id` is
`fuel_cost_shock_event_21d` (the derived 0/1 event series) — that's the series the harness needs
for warmup checks and outcome resolution, but it's the wrong series to build a price-history
prompt payload from. The fix: always read price context from `FUEL_SERIES_ID` directly,
independent of whatever series the task is being *scored* against. Only the binary shock task
(`TASK_SHOCK_SPEC`) is re-declared here — the backtest specs only cover the primary binary task,
not the secondary trajectory task from `01_...`.


In [ ]:
FUEL_FORECASTER_INSTRUCTION = """
## Role

You are an expert jet-fuel and crude-oil market analyst producing calibrated forecasts to support airline hedging, budgeting, and financial risk decisions.

## Input

You will receive a JSON payload containing:
- `task_spec`: the exact question and required JSON output schema
- `as_of`: the forecast origin date (temporal cutoff)
- `horizons`: integer horizon steps (business days ahead)
- `standard_quantiles`: quantile levels for continuous forecasts (when applicable)
- `origin_price_usd_gal`: jet-fuel proxy (heating oil futures) close on the origin date
- `target_history_csv`: compressed jet-fuel proxy daily close history

Call ``search_web`` BEFORE answering -- it is your only source of qualitative market intelligence (OPEC+ policy, geopolitical supply risk, EIA releases); you have no other tools and no post-training-cutoff knowledge to rely on.

## Output contract

Read the data and the search briefing carefully, then execute the task in `task_spec` precisely.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response` -- the exact schema is described in `task_spec`. Otherwise return the JSON directly as plain text with no preamble.
""".strip()


def build_fuel_forecaster_config(
    model: str = AGENT_MODEL,
    search_model: str = SEARCH_MODEL,
    verifier_model: str = VERIFIER_MODEL,
) -> AgentConfig:
    """Build the Forecaster Agent's :class:`AgentConfig`, wiring in the News Agent."""
    return AgentConfig(
        name="fuel_shock_forecaster",
        model=model,
        instruction=FUEL_FORECASTER_INSTRUCTION,
        context_retrieval=build_fuel_context_retrieval_config(search_model=search_model, verifier_model=verifier_model),
    )


def _compress_history(df: pd.DataFrame) -> str:
    """Compress daily price history: recent 6 months daily, older weekly averages."""
    frame = df.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    cutoff = frame["timestamp"].max() - pd.DateOffset(months=6)

    recent = frame[frame["timestamp"] >= cutoff]
    old = frame[frame["timestamp"] < cutoff]

    rows: list[str] = ["date,close"]
    if not old.empty:
        weekly = old.set_index("timestamp")["value"].resample("W").mean().dropna()
        rows.extend(f"{date.date()},{val:.3f}" for date, val in weekly.items())
    rows.extend(f"{row['timestamp'].date()},{row['value']:.3f}" for _, row in recent.iterrows())
    return "\n".join(rows)


class FuelMultitaskPromptBuilder(BaseModel):
    """Prompt builder for task-spec-driven Forecaster Agent calls.

    Always builds its price-context payload from ``FUEL_SERIES_ID``, regardless of
    ``task.target_series_id`` -- the task's target series is whatever the harness scores against
    (the price series for 01_...'s live demo, the derived event series for the backtest spec),
    but the LLM always needs the same jet-fuel price history to reason about either question.
    """

    task_spec: str

    model_config = {"extra": "forbid"}

    def __call__(self, *, task: ForecastingTask, context: ForecastContext) -> str:
        df = context.get_series(FUEL_SERIES_ID)
        last_row = df.iloc[-1]
        payload = {
            "task": task.task_id,
            "task_spec": self.task_spec,
            "as_of": str(context.as_of)[:10],
            "horizons": list(task.horizons),
            "standard_quantiles": list(STANDARD_QUANTILES),
            "origin_price_usd_gal": float(last_row["value"]),
            "target_history_csv": _compress_history(df),
        }
        return json.dumps(payload, indent=2)


TASK_SHOCK_SPEC = (
    f"Estimate the probability that the jet-fuel proxy price will close MORE\n"
    f"THAN {SHOCK_THRESHOLD_PCT:.0%} HIGHER than today's price at the end of\n"
    f"{SHOCK_HORIZON_DAYS} trading days (the cost-shock event).\n\n"
    "This is a directional upside question only.\n\n"
    "Calibration guidance:\n"
    "  - No unusual upside catalyst       -> base rate ~10-15%\n"
    "  - Escalating unconfirmed risk      -> 20-40%\n"
    "  - Confirmed supply disruption      -> 60-85%\n\n"
    "Use `key_signals` to name the specific drivers (OPEC+ decisions, shipping-lane\n"
    "risk, inventory levels, refinery outages, macro demand) behind your estimate --\n"
    "this is the primary place downstream readers look for the 'why'.\n\n"
    "If a `set_model_response` tool is available, call it with your complete "
    "JSON as `json_response`. Otherwise return the JSON directly as plain text.\n\n"
    "Required JSON format:\n" + DiscreteAgentForecastOutput.prompt_schema_json()
)

forecaster_config = build_fuel_forecaster_config()
shock_prompt_builder = FuelMultitaskPromptBuilder(task_spec=TASK_SHOCK_SPEC)
agent_predictor = AgentPredictor(
    agent_config=forecaster_config,
    prompt_builder=shock_prompt_builder,
    output_schema=DiscreteAgentForecastOutput,
)
print(f"Forecaster Agent config: {forecaster_config.name} (model={forecaster_config.model})")
print(f"Agent predictor id: {agent_predictor.predictor_id}")


---
## 4. Baselines

Three conventional/naive predictors, all implementing the same `Predictor` API as the agent, so
they drop into the same backtest loop below with no special-casing:

| Predictor | What it sees | Notes |
|---|---|---|
| `HistoricalFrequencyPredictor` | Past outcomes only | Constant climatological base rate — the floor every other predictor must clear |
| `GARCHPredictor` | Jet-fuel proxy price history only | GARCH(1,1) volatility model fit at every origin; converts fitted drift/variance into P(21-day return > 10%) under a Gaussian tail assumption |
| `LogisticRegressionBaseline` | Jet-fuel proxy + WTI + USD-index trailing returns/volatility | Logistic regression refit at every origin on leak-safe trailing-return/volatility features |

`GARCHPredictor` and `LogisticRegressionBaseline` are new, generic additions to
`aieng.forecasting.methods.baselines` (not fuel-specific — they take their price/covariate series
ids as constructor arguments), added by this change to close the gap flagged in
`README.md`'s "baselines not yet implemented" note.


In [ ]:
historical_freq_baseline = HistoricalFrequencyPredictor()
garch_baseline = GARCHPredictor(price_series_id=FUEL_SERIES_ID, threshold_pct=SHOCK_THRESHOLD_PCT)
logreg_baseline = LogisticRegressionBaseline(
    price_series_id=FUEL_SERIES_ID,
    covariate_series_ids=[WTI_SERIES_ID, USD_INDEX_SERIES_ID],
)

all_predictors = [agent_predictor, historical_freq_baseline, garch_baseline, logreg_baseline]
PREDICTOR_LABELS = {
    agent_predictor.predictor_id: "Forecaster Agent",
    historical_freq_baseline.predictor_id: "Historical frequency",
    garch_baseline.predictor_id: "GARCH(1,1)",
    logreg_baseline.predictor_id: "Logistic regression",
}
for p in all_predictors:
    print(f"  {p.predictor_id}  ({PREDICTOR_LABELS[p.predictor_id]})")


---
## 5. Smoke test (3 origins)

`specs/fuel_shock_smoke.yaml` covers 3 monthly origins from June-July 2025 -- cheap enough to run
every time, and (per the header note added to that spec) already post-cutoff so it exercises the
same leakage-fence code path as the real backtest. Run this before the full backtest below to
confirm the pipeline runs end to end against historical origins.

`cached_backtest` writes each result to `data/predictions/<spec_id>/<predictor_id>.yaml` and
reuses it on later runs; pass `force_refresh=True` to recompute.


In [ ]:
with (SPECS_DIR / "fuel_shock_smoke.yaml").open() as f:
    smoke_spec = BacktestSpec.model_validate(yaml.safe_load(f))

print(f"Smoke spec origins: {[o.date() for o in smoke_spec.origins()]}")

smoke_results = {}
for predictor in all_predictors:
    print(f"Running {predictor.predictor_id} (smoke) ...", flush=True)
    smoke_results[predictor.predictor_id] = cached_backtest(
        predictor=predictor,
        spec=smoke_spec,
        spec_id="fuel_shock_smoke",
        data_service=data_service,
        store_dir=PREDICTIONS_DIR,
    )
    r = smoke_results[predictor.predictor_id]
    print(f"  mean Brier = {r.mean_score:.4f}  ({len(r.predictions)} predictions, {r.skipped_origins} skipped)")


---
## 6. Full post-cutoff backtest (~14 origins)

`specs/fuel_shock_backtest.yaml` covers monthly origins from **2025-06-02 to 2026-08-03** — every
origin postdates the ~January 2025 model cutoff (see the spec file's header for the full
rationale). This is the run that actually answers "is the agent forecasting or memorizing".


In [ ]:
with (SPECS_DIR / "fuel_shock_backtest.yaml").open() as f:
    backtest_spec = BacktestSpec.model_validate(yaml.safe_load(f))

print(f"Backtest spec origins ({len(backtest_spec.origins())}): {[o.date() for o in backtest_spec.origins()]}")

backtest_results = {}
for predictor in all_predictors:
    print(f"Running {predictor.predictor_id} (full backtest) ...", flush=True)
    backtest_results[predictor.predictor_id] = cached_backtest(
        predictor=predictor,
        spec=backtest_spec,
        spec_id="fuel_shock_backtest",
        data_service=data_service,
        store_dir=PREDICTIONS_DIR,
    )
    r = backtest_results[predictor.predictor_id]
    print(f"  mean Brier = {r.mean_score:.4f}  ({len(r.predictions)} predictions, {r.skipped_origins} skipped)")


---
## 7. Leaderboard & calibration

`skill_vs_reference` is `1 - mean_score / reference_mean_score` against
`HistoricalFrequencyPredictor`: positive means the predictor beats the naive base rate, 0 matches
it, negative means it's worse than knowing nothing. The calibration table bins the Forecaster
Agent's predicted probabilities and compares them to the realised outcome frequency in each bin
(reliability diagram data) — a well-calibrated agent should show `mean_predicted` close to
`mean_observed` in every bin.


In [ ]:
def score_leaderboard(results: dict, reference_id: str) -> pd.DataFrame:
    """Mean-score leaderboard with skill score against a reference predictor."""
    reference_score = results[reference_id].mean_score
    rows = []
    for predictor_id, result in results.items():
        skill = 1.0 - result.mean_score / reference_score if reference_score else float("nan")
        rows.append(
            {
                "predictor_id": predictor_id,
                "label": PREDICTOR_LABELS.get(predictor_id, predictor_id),
                "mean_brier": result.mean_score,
                "n_predictions": len(result.predictions),
                "skipped_origins": result.skipped_origins,
                "skill_vs_reference": skill,
            }
        )
    return pd.DataFrame(rows).sort_values("mean_brier")


def shock_calibration_table(probabilities: list[float], outcomes: list[float], n_bins: int = 5) -> pd.DataFrame:
    """Bin predicted probabilities and compare to realised outcome frequency (reliability diagram data)."""
    df = pd.DataFrame({"probability": probabilities, "outcome": outcomes}).dropna()
    if df.empty:
        return pd.DataFrame(columns=["bin", "mean_predicted", "mean_observed", "n"])
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    df["bin"] = pd.cut(df["probability"], bins=edges, include_lowest=True)
    grouped = df.groupby("bin", observed=True).agg(
        mean_predicted=("probability", "mean"), mean_observed=("outcome", "mean"), n=("outcome", "count")
    )
    return grouped.reset_index()


board = score_leaderboard(backtest_results, reference_id=historical_freq_baseline.predictor_id)
print(board.to_string(index=False))

agent_result = backtest_results[agent_predictor.predictor_id]
agent_probs = [p.payload.probability for p in agent_result.predictions]

# Resolve each prediction's realised outcome the same way the harness itself does (by
# forecast_date, against the full unfiltered series), so calibration matches what was scored.
full_event_df = data_service.get_series(SHOCK_EVENT_SERIES_ID, as_of=naive_utc_now())
resolved_outcomes = []
for p in agent_result.predictions:
    match = full_event_df[pd.to_datetime(full_event_df["timestamp"]) == pd.Timestamp(p.forecast_date)]
    resolved_outcomes.append(float(match["value"].iloc[0]) if not match.empty else float("nan"))

print()
print("Agent calibration (predicted probability vs. realised outcome frequency):")
print(shock_calibration_table(agent_probs, resolved_outcomes).to_string(index=False))


---
## 8. Leakage-fence spot check

The News Agent's `search_web` tool logs its `effective_cutoff` at INFO level every time it runs
(`aieng.forecasting.methods.agentic.agent_factory`). Raising that logger to INFO before re-running
the smoke backtest (with `force_refresh=True` so it actually calls the agent instead of loading
the cache from §5) prints one line per search call showing the cutoff that was actually enforced
-- confirm it matches each origin's `as_of` date, not today's date, and that
`[SEARCH_VERIFICATION_FAILED]` doesn't appear on every call (which would mean the news agent could
never find verifiably pre-cutoff information for that query).


In [ ]:
logging.basicConfig(level=logging.WARNING)
logging.getLogger("aieng.forecasting.methods.agentic.agent_factory").setLevel(logging.INFO)

spot_check_result = cached_backtest(
    predictor=agent_predictor,
    spec=smoke_spec,
    spec_id="fuel_shock_smoke",
    data_service=data_service,
    store_dir=PREDICTIONS_DIR,
    force_refresh=True,
)
print(f"Spot-check run: mean Brier = {spot_check_result.mean_score:.4f}")
print("Scroll up through the log output above -- every 'search_web: skipping cutoff enforcement'")
print("or 'search_web verification attempt' line should show effective_cutoff matching one of:")
print(f"  {[o.date() for o in smoke_spec.origins()]}")


---
## 9. Caveats and next steps

- **Untuned baselines.** `GARCHPredictor` and `LogisticRegressionBaseline` use deliberately
  simple, untuned configurations (constant-mean GARCH(1,1); a handful of trailing-return/vol
  features) — they're reference points, not competition entries.
- **No run-budget protection.** Unlike `boc_rate_decisions`'s protected post-2025 eval (which
  uses `EvalSpec` + `EvalTracker` to cap re-runs against a held-out window), this backtest uses
  the plain `BacktestSpec`/`cached_backtest` harness with no run cap. Because the *entire* window
  here is already past the cutoff (there's no separate "pedagogical, pre-cutoff" backtest the way
  BoC has), repeated re-runs while iterating on the agent's prompt are themselves a mild form of
  overfitting to this specific window. If this evolves into a longer-running project, moving
  `fuel_shock_backtest.yaml` to an `EvalSpec` with `EvalTracker`-enforced `max_runs` (mirroring
  `boc_rate_decisions/02_boc_rate_direction_experiment.ipynb` §10) would close that gap.
- **Single run per predictor.** No self-consistency / median-over-N-runs sampling (the
  `01_...` README's noted stretch goal) — each origin is one LLM call.
